# Racecar — kinematic MPC on the dynamic 1/10 car

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/alx87grd/minilink/blob/main/examples/teaching/topics/optimal_control/racecar_mpc.ipynb)

Same kinematic MPC, now over a PID speed loop on the dynamic 1/10 racecar. The planner sees a bicycle $x = [x,\, y,\, \theta]$, $u = [v,\, \delta]$. The plant is the 11-state car: steer is direct, speed goes through a PID on drive power.

This page uses the [minilink](https://github.com/alx87grd/minilink) toolbox.


In [ ]:
# Local conda: minilink already installed. Colab: clone + path + meshcat.
import sys

if "google.colab" in sys.modules:
    get_ipython().run_line_magic("matplotlib", "inline")
    get_ipython().system("git clone https://github.com/alx87grd/minilink")
    sys.path.insert(0, "/content/minilink")
    get_ipython().system("pip install -q meshcat")

In [ ]:
import numpy as np

from minilink import (
    PID,
    Demux,
    DiagramSystem,
    PlanningProblem,
    QuadraticCost,
    TrajectoryOptimizationPlanner,
    UdeSRacecar,
    UdeSRacecarDyn3D,
)
from minilink.control.mpc import ModelPredictiveController, mpc_animation_overlays
from minilink.core.hybrid_composition import hybrid_closed_loop
from minilink.graphical.catalog.racecar_skin import racecar_skin_2d, racecar_skin_3d
from minilink.planning import (
    ReferenceTrack,
    Scene,
    Sphere,
    bind,
    circuit_waypoints,
    from_waypoints,
    point_probe,
    quadratic_hinge,
)

MU = 0.4  # [-] floor grip; drop toward 0.4 and the rear saturates in the corners
V_REF = 2.0  # [m/s] the 1 m corners hold this at MU=1, not at MU=0.4
TF = 20.0
MPC_DT = 0.05
MPC_HORIZON = 1.8
KP, KI, KD, TAU = 40.0, 20.0, 5.0, 0.05  # PID on drive power
P_CRUISE = 10.0  # [W] initial throttle

## Circuit

Same rounded rectangle as the kinematic demo, with cones on the far straight. `ReferenceTrack` is the lane (`half_width=0.6`); `Scene` holds the cones.


In [ ]:
path = circuit_waypoints(length=6.0, width=4.0, radius=1.0)
track = ReferenceTrack(from_waypoints(path), half_width=0.6)
scene = Scene(
    obstacles=[
        Sphere((0.0, 1.8), 0.15),
        Sphere((1.0, 2.0), 0.15),
        Sphere((2.0, 2.2), 0.15),
        Sphere((-2.2, 2.3), 0.35),
    ]
)

## Design model

The MPC plans on the kinematic bicycle `UdeSRacecar`. Inputs are speed and steer. The start is a little off the line: path distance has no derivative on it.


In [ ]:
model = UdeSRacecar()
model.inputs["u"].lower_bound = np.array([0.0, -0.52])
model.inputs["u"].upper_bound = np.array([5.0, 0.52])

start = path[0]
heading = np.arctan2(path[1, 1] - start[1], path[1, 0] - start[0])
x0 = np.array(
    [start[0] - 0.1 * np.sin(heading), start[1] + 0.1 * np.cos(heading), heading]
)
model.x0 = x0

## Plant

The 11-state car (`UdeSRacecarDyn3D`): rolling angles spin the wheels; `mu` is the grip.


In [ ]:
plant = UdeSRacecarDyn3D()
plant.params["mu"] = MU
plant.camera_follow_frame = None
plant.camera_scale = 4.0
plant.x0 = np.array(
    [
        x0[0],
        x0[1],
        x0[2],
        V_REF,
        0.0,
        0.0,
        V_REF / plant.params["r_r"],
        0.0,
        P_CRUISE,
        0.0,
        0.0,
    ]
)

## Inner loop

MPC command $[v,\, \delta]$ in: steer is direct, speed goes through a PID on drive power. The diagram output is the pose $y[0:3]$.


In [ ]:
inner = DiagramSystem()
inner.add_subsystem(Demux((1, 1)), "cmd")
inner.add_subsystem(
    PID(
        Kp=KP,
        Ki=KI,
        Kd=KD,
        tau=TAU,
        ports="reference",
        u_min=-plant.params["P_max"],
        u_max=plant.params["P_max"],
    ),
    "speed",
)
inner.add_subsystem(plant, "car")
inner.add_subsystem(Demux((3, 8), port="y"), "pose")
inner.add_input_port("u", dim=2)
inner.connect("input", "u", "cmd", "u")
inner.connect("cmd", "u[0]", "speed", "r")
inner.connect("car", "speed", "speed", "y")
inner.connect("speed", "u", "car", "P_cmd")
inner.connect("cmd", "u[1]", "car", "delta_cmd")
inner.connect("car", "y", "pose", "y")
inner.connect_new_output_port("pose", "y[0:3]", "y")

In [ ]:
inner.plot_diagram()

## Cost

Same three terms as the kinematic demo: cruise, stay in the lane, miss the cones.


In [ ]:
probe = bind(model, point_probe())
quad = QuadraticCost.from_system(
    model, Q=np.zeros((3, 3)), R=np.diag([2.0, 1.0]), ubar=np.array([V_REF, 0.0])
)
corridor = track.corridor_field(probe).as_cost(weight=20.0, shaping=quadratic_hinge())
obstacle = scene.clearance_field(probe).as_cost(
    weight=40.0, shaping=quadratic_hinge(threshold=0.2)
)
cost = quad + corridor + obstacle

## MPC

Receding-horizon collocation on the kinematic design.


In [ ]:
planner = TrajectoryOptimizationPlanner(
    PlanningProblem(sys=model, x_start=x0, cost=cost, tf=MPC_HORIZON),
    n_steps=20,
    transcription="direct_collocation",
    compile_backend="jax",
    optimizer_method="scipy_slsqp",
    optimizer_options={"maxiter": 40, "ftol": 0.05},
)
mpc = ModelPredictiveController(planner, dt_mpc=MPC_DT, warm_start=True, verbose=True)

## Closed loop

Hybrid, because the inner plant is a diagram. The computer ticks every `dt_mpc`; the inner plant integrates in between.


In [ ]:
computer = mpc.export_to_computer()
diagram = hybrid_closed_loop(
    computer.diagram,
    inner,
    schedule=computer.schedule,
    computer=computer,
    computer_out="u_ff",
    computer_in="y",
    plant_in="u",
    plant_out="y",
)

In [ ]:
diagram.plot_diagram()

## Lap

`result.plant` is the inner diagram, whose first states are the PID, not $(x, y)$. `trajectory_of` pulls the car pose for the overlays.


In [ ]:
result = diagram.compute_trajectory(tf=TF, plant_dt_inner=0.002, compile_backend="jax")
car_run = inner.trajectory_of(plant, result.plant)
overlays = mpc_animation_overlays(
    result,
    planner,
    scene=scene,
    track=track,
    traj=car_run,
    trail=False,
)

In [ ]:
diagram.plot_trajectory(signals=("car:speed", "car:grip", "speed:u"))

## Animate

The extra plant states are the wheel rolling angles. Same lap as the script: Meshcat, 3-D skin.


In [ ]:
plant.skin = racecar_skin_3d
diagram.animate(
    overlays=overlays,
    renderer="meshcat",
    is_3d=True,
    native=False,
    # native=True,
)